# AgroSmart Puno · UNIDAD 3 — Integración, DataOps y hardening (S13 → S14)
**Equipo:** Jose Armando Mamani Poccohuanca (humedad del suelo) y Mijael Yurins Coaquira Justo (temperatura, Azángaro)
**Producto:** una **ejecución integrada** de punta a punta (batch + streaming + modelo + observabilidad) para **las dos dimensiones**, con compuertas previas y de calidad, KPI con percentiles, publicación en Prometheus, manifiesto de ejecución y revisión técnica final (hardening).

| Bloque | Sesión | Sección |
|---|---|---|
| Compuertas previas | S13 | 1 |
| Batch: KPI con percentiles y compuertas de calidad/plausibilidad | S13 | 2 |
| Streaming y error de predicción en producción | S13 | 3 |
| Publicación en Prometheus + registro | S13 | 4 |
| Manifiesto y comparación de dos corridas | S13 | 5 |
| Compuerta que falla a propósito | S13 | 6 |
| Revisión técnica: lista de comprobación, no regresión, riesgos, demo | S14 | 7 |

> **Antes de ejecutar:** Kafka, PostgreSQL+Jupyter, Prometheus/Grafana, los dos consumers Spark y los productores de 2 minutos deben estar arriba (ver `GUIA_EJECUCION_CMD.md`) y los modelos entrenados. Una ejecución que termina sin error **no** demuestra que el resultado sea correcto (caso GitLab 2017): por eso existen las compuertas.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, time, platform, hashlib, urllib.request
import pandas as pd
import psycopg2
from IPython.display import display

REPO = Path('/home/jovyan/agrosmart-repo')
RAIZ = Path('/home/jovyan/work')
EVID = RAIZ/'U3_Integracion'/'evidencias'; EVID.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); INICIO = time.time()
PARAM_DB = dict(host=os.getenv('POSTGRES_HOST', 'postgres'), dbname=os.getenv('POSTGRES_DB', 'agrosmart_u2'),
                user=os.getenv('POSTGRES_USER', 'agrosmart'), password=os.getenv('POSTGRES_PASSWORD', ''), connect_timeout=10)
KAFKA = os.getenv('KAFKA_BOOTSTRAP', 'kafka:9092')
ORIGENES = "('open_meteo_modelado','open_meteo_interpolado_2min')"      # flujo vivo (excluye la carga inicial simulada)
VENTANA_STREAM_S = 150            # observación del flujo en vivo (cubre >= 1 evento del productor de 2 min)
UMBRAL_P50_S, UMBRAL_P95_S = 60, 300     # plausibilidad: latencia típica < 60 s; cola > 300 s = advertencia

def sql(q, params=None):
    with psycopg2.connect(**PARAM_DB) as con, con.cursor() as cur:
        cur.execute(q, params)
        if cur.description is None: return None
        return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

with psycopg2.connect(**PARAM_DB) as con, con.cursor() as cur:
    for f in ('schema.sql', 'schema_temperatura.sql', 'schema_integracion.sql'): cur.execute((REPO/'pyspark'/'sql'/f).read_text(encoding='utf-8'))

manifiesto = {'runId': RUN_ID, 'equipo': ['Jose Armando Mamani Poccohuanca', 'Mijael Yurins Coaquira Justo'],
              'python': platform.python_version(), 'plataforma': platform.platform(), 'compuertas': {}, 'batch': {}, 'streaming': {}, 'observabilidad': {}, 'advertencias': []}
print('Ejecución integrada', RUN_ID)

Ejecución integrada 20261120T060237Z


---
# 1 · Compuertas previas (preflight)
Comprueban **antes** de procesar que las dependencias existen y responden. Si una falla, la ejecución **se detiene con el nombre de la dependencia** (en vez de fallar minutos después dentro de Spark).

In [2]:
def http_ok(url, timeout=6):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r: return r.status == 200
    except Exception: return False

def kafka_topicos():
    from kafka import KafkaAdminClient
    adm = KafkaAdminClient(bootstrap_servers=KAFKA, request_timeout_ms=8000); t = set(adm.list_topics()); adm.close(); return t

def tablas_ok():
    t = set(sql("SELECT table_name FROM information_schema.tables WHERE table_schema='public'").table_name)
    return {'eventos_telemetria','predicciones_humedad','ventanas_humedad','metricas_modelo','metricas_pipeline','eventos_temperatura',
            'predicciones_temperatura','ventanas_temperatura','metricas_modelo_temperatura','ejecuciones_integracion'} <= t

def consumer_activo(consulta):
    d = sql("SELECT COUNT(*) n FROM metricas_pipeline WHERE consulta=%s AND activa AND error IS NULL AND observado_utc > NOW() - INTERVAL '90 seconds'", (consulta,))
    return int(d.n.iloc[0]) == 1

TOPICOS = {'agrosmart-u2-humedad-telemetria', 'agrosmart-u2-humedad-empresarial', 'agrosmart-u2-temperatura-telemetria', 'agrosmart-u2-temperatura-empresarial'}
PREVIAS = {
 'PostgreSQL responde y las 10 tablas existen': tablas_ok,
 'Kafka responde y los 4 tópicos existen': lambda: TOPICOS <= kafka_topicos(),
 'Modelos de José (humedad 1/2/3 h) existen': lambda: all((RAIZ/'Sensorhumedad'/'modelos'/f'humedad_{h}h').exists() for h in (1, 2, 3)),
 'Modelo de Mijael (temperatura 1 h) existe': lambda: (RAIZ/'Sensortemperatura'/'modelos'/'temperatura_1h').exists(),
 'Parquet Gold de U1 (José y Mijael) existe': lambda: (RAIZ/'U1_Batch_Humedad'/'artifacts'/'humedad_gold_particionado').exists() and (RAIZ/'U1_Batch_Temperatura'/'artifacts'/'temperatura_gold_particionado').exists(),
 'Prometheus listo': lambda: http_ok('http://prometheus:9090/-/ready'),
 'Pushgateway listo': lambda: http_ok('http://pushgateway:9091/-/ready'),
 'Grafana saludable': lambda: http_ok('http://grafana:3000/api/health'),
 'Consumer Spark de José activo (heartbeat < 90 s)': lambda: consumer_activo('telemetria'),
 'Consumer Spark de Mijael activo (heartbeat < 90 s)': lambda: consumer_activo('temp_telemetria'),
}
def correr_compuertas(compuertas):
    res = {}
    for nombre, fn in compuertas.items():
        try: ok = bool(fn())
        except Exception as e: ok = False
        res[nombre] = ok; print(('OK    ' if ok else 'FALLA ') + nombre)
    return res

def registrar_falla(etapa):
    # Una ejecución que se detiene también se registra (manifiesto + fila FALLA): así el tablero no muestra una corrida vieja como vigente.
    try:
        manifiesto.update({'estado': 'FALLA', 'etapaFalla': etapa, 'duracionTotalS': round(time.time() - INICIO, 1)})
        (EVID/f'manifiesto_{RUN_ID}_FALLA.json').write_text(json.dumps(manifiesto, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
        tot = sum(len(v) for v in manifiesto['compuertas'].values()); ok = sum(sum(v.values()) for v in manifiesto['compuertas'].values())
        with psycopg2.connect(**PARAM_DB) as con, con.cursor() as cur:
            cur.execute("INSERT INTO ejecuciones_integracion (run_id,estado,compuertas_ok,compuertas_total,duracion_s,manifiesto) VALUES (%s,'FALLA',%s,%s,%s,%s) ON CONFLICT(run_id) DO NOTHING",
                        (RUN_ID + '-FALLA', ok, tot, manifiesto['duracionTotalS'], json.dumps(manifiesto, default=str)))
    except Exception as e:
        print('No se pudo registrar la falla:', e)

def detener_si_falla(res, etapa, registrar=True):
    falla = [n for n, ok in res.items() if not ok]
    if falla:
        if registrar: registrar_falla(etapa)
        raise AssertionError(f'[{etapa}] Compuertas que fallan: {falla}')

manifiesto['compuertas']['previas'] = correr_compuertas(PREVIAS)
detener_si_falla(manifiesto['compuertas']['previas'], 'previas')
print('\nTodas las compuertas previas pasan: la ejecución puede empezar.')

OK    PostgreSQL responde y las 10 tablas existen
OK    Kafka responde y los 4 tópicos existen
OK    Modelos de José (humedad 1/2/3 h) existen
OK    Modelo de Mijael (temperatura 1 h) existe
OK    Parquet Gold de U1 (José y Mijael) existe
OK    Prometheus listo
OK    Pushgateway listo
OK    Grafana saludable
OK    Consumer Spark de José activo (heartbeat < 90 s)
OK    Consumer Spark de Mijael activo (heartbeat < 90 s)

Todas las compuertas previas pasan: la ejecución puede empezar.


---
# 2 · Batch: KPI con percentiles y compuertas de calidad y de plausibilidad
**KPI de tiempo del dominio: latencia del pipeline** = `fecha_ingesta − emitido_utc` (segundos desde que el productor emite el evento hasta que Spark lo guarda), sobre el flujo **vivo** de las últimas 24 h (la carga inicial simulada se excluye).
Se publican **p50** (evento típico) y **p95** (la cola), **nunca solo el promedio**: un promedio puede describir un valor que no representa a ningún grupo de eventos. La plausibilidad se verifica sobre el **p50** (< 60 s); el p95 queda como **advertencia** (> 300 s).

In [3]:
VENTANA_KPI_MIN = 30   # el KPI mide eventos EMITIDOS en los últimos 30 min

# 2.0 Diagnóstico: latencia media por hora de emisión (revela si hubo un atraso acumulado en Kafka)
for dim, (tabla, ing) in {'humedad': ('eventos_telemetria', 'ingestado_utc'), 'temperatura': ('eventos_temperatura', 'fecha_ingesta')}.items():
    print(f'--- {dim}: latencia por hora de emisión (últimas 24 h) ---')
    display(sql(f'''SELECT date_trunc('hour', emitido_utc) AS hora_emision, COUNT(*) eventos,
                      ROUND(AVG(EXTRACT(EPOCH FROM ({ing} - emitido_utc)))::numeric, 1) lat_media_s,
                      ROUND(MAX(EXTRACT(EPOCH FROM ({ing} - emitido_utc)))::numeric, 1) lat_max_s
               FROM {tabla} WHERE origen IN {ORIGENES} AND emitido_utc IS NOT NULL AND emitido_utc >= NOW() - INTERVAL '24 hours'
               GROUP BY 1 ORDER BY 1'''))

def kpi_latencia(tabla, ing):
    d = sql(f'''SELECT COUNT(*) n, AVG(lat) promedio, percentile_cont(0.5) WITHIN GROUP (ORDER BY lat) p50,
                       percentile_cont(0.95) WITHIN GROUP (ORDER BY lat) p95, MAX(lat) maximo, AVG((lat > 60)::int) frac_mayor_1min
                FROM (SELECT EXTRACT(EPOCH FROM ({ing} - emitido_utc)) AS lat FROM {tabla}
                      WHERE origen IN {ORIGENES} AND emitido_utc IS NOT NULL AND emitido_utc >= NOW() - make_interval(mins => {VENTANA_KPI_MIN})) t''').iloc[0]
    return {k: (None if pd.isna(v) else float(v)) for k, v in d.items()}

dims = {'humedad': ('eventos_telemetria', 'humedad_suelo_pct', 0, 100, 'ingestado_utc'), 'temperatura': ('eventos_temperatura', 'temperatura_c', -60, 60, 'fecha_ingesta')}
kpi = {dim: kpi_latencia(t, ing) for dim, (t, _c, _lo, _hi, ing) in dims.items()}
calidad = {}
for dim, (tabla, col, lo, hi, ing) in dims.items():
    k = kpi[dim]
    nulos = int(sql(f"SELECT COUNT(*) n FROM {tabla} WHERE {col} IS NULL OR fecha_evento IS NULL").n.iloc[0])
    fuera = int(sql(f"SELECT COUNT(*) n FROM {tabla} WHERE {col} NOT BETWEEN {lo} AND {hi}").n.iloc[0])
    dup = int(sql(f"SELECT COUNT(*)-COUNT(DISTINCT evento_id) n FROM {tabla}").n.iloc[0])
    calidad[dim] = {f'[{dim}] hay eventos del flujo vivo ({VENTANA_KPI_MIN} min)': (k['n'] or 0) > 0,
                    f'[{dim}] sin nulos en {col}/fecha_evento': nulos == 0,
                    f'[{dim}] {col} dentro de [{lo}, {hi}]': fuera == 0,
                    f'[{dim}] sin evento_id duplicado': dup == 0,
                    f'[{dim}] PLAUSIBILIDAD: latencia p50 < {UMBRAL_P50_S} s': (k['p50'] is not None and k['p50'] < UMBRAL_P50_S)}
    if k['p95'] is not None and k['p95'] > UMBRAL_P95_S:
        manifiesto['advertencias'].append(f'[{dim}] p95 de latencia {k["p95"]:.0f} s > {UMBRAL_P95_S} s: el promedio no representa a la mayoría de los eventos')
display(pd.DataFrame(kpi).T.round(3))
todas_cal = {n: ok for d in calidad.values() for n, ok in d.items()}
manifiesto['compuertas']['calidad'] = correr_compuertas({n: (lambda ok=ok: ok) for n, ok in todas_cal.items()})
manifiesto['batch'] = {'kpiLatenciaPipeline': kpi}
for w in manifiesto['advertencias']: print('ADVERTENCIA:', w)
detener_si_falla(manifiesto['compuertas']['calidad'], 'calidad')

--- humedad: latencia por hora de emisión (últimas 24 h) ---


,hora_emision,eventos,lat_media_s,lat_max_s
0,2026-11-19 01:00:00-05:00,29,4.8,6.6
1,2026-11-19 02:00:00-05:00,30,4.4,6.6
2,2026-11-19 03:00:00-05:00,30,4.2,6.5
3,2026-11-19 04:00:00-05:00,30,4.6,6.6
4,2026-11-19 05:00:00-05:00,30,4.2,6.6
5,2026-11-19 06:00:00-05:00,30,5.1,6.5
6,2026-11-19 07:00:00-05:00,30,5.5,6.6
7,2026-11-19 08:00:00-05:00,30,4.8,6.2
8,2026-11-19 09:00:00-05:00,30,5.6,7.2
9,2026-11-19 10:00:00-05:00,4,6.1,6.3


--- temperatura: latencia por hora de emisión (últimas 24 h) ---


,hora_emision,eventos,lat_media_s,lat_max_s
0,2026-11-19 01:00:00-05:00,28,4.0,6.3
1,2026-11-19 02:00:00-05:00,31,3.3,6.2
2,2026-11-19 03:00:00-05:00,30,4.6,6.2
3,2026-11-19 04:00:00-05:00,30,4.8,6.2
4,2026-11-19 05:00:00-05:00,30,3.9,6.3
5,2026-11-19 06:00:00-05:00,30,4.8,6.2
6,2026-11-19 07:00:00-05:00,30,5.2,6.2
7,2026-11-19 08:00:00-05:00,30,4.5,6.0
8,2026-11-19 09:00:00-05:00,30,5.3,6.8
9,2026-11-19 10:00:00-05:00,4,5.7,5.9


,n,promedio,p50,p95,maximo,frac_mayor_1min
humedad,15.0,6.568,6.529,9.425,13.967,0.0
temperatura,14.0,5.536,6.126,7.311,7.478,0.0


OK    [humedad] hay eventos del flujo vivo (30 min)
OK    [humedad] sin nulos en humedad_suelo_pct/fecha_evento
OK    [humedad] humedad_suelo_pct dentro de [0, 100]
OK    [humedad] sin evento_id duplicado
OK    [humedad] PLAUSIBILIDAD: latencia p50 < 60 s
OK    [temperatura] hay eventos del flujo vivo (30 min)
OK    [temperatura] sin nulos en temperatura_c/fecha_evento
OK    [temperatura] temperatura_c dentro de [-60, 60]
OK    [temperatura] sin evento_id duplicado
OK    [temperatura] PLAUSIBILIDAD: latencia p50 < 60 s


## 2.1 Capa Gold (Parquet particionado) leída con Spark
Comprueba que la salida analítica de la Unidad 1 de **cada integrante** sigue legible y consistente (filas, rango y particiones `anio_mes`).

In [4]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master('local[2]').appName('AgroSmart-U3-Integracion').config('spark.sql.shuffle.partitions', '2').config('spark.sql.session.timeZone', 'UTC').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
gold = {}
for dim, ruta, col in (('humedad', RAIZ/'U1_Batch_Humedad'/'artifacts'/'humedad_gold_particionado', 'humedad_suelo_pct'),
                       ('temperatura', RAIZ/'U1_Batch_Temperatura'/'artifacts'/'temperatura_gold_particionado', 'temperatura_c')):
    df = spark.read.parquet(str(ruta)); n = df.count(); r = df.agg(F.min(col), F.max(col)).first()
    parts = sorted(p.name for p in ruta.iterdir() if p.name.startswith('anio_mes'))
    gold[dim] = {'filas': n, 'min': r[0], 'max': r[1], 'particiones': parts}
    print(f'{dim:12s} filas={n:,} rango=[{r[0]:.2f}, {r[1]:.2f}] particiones={parts}')
manifiesto['batch']['parquetGold'] = gold
assert all(g['filas'] > 0 for g in gold.values()), 'Parquet Gold vacío'

humedad      filas=1,392 rango=[6.00, 37.90] particiones=['anio_mes=2026-09', 'anio_mes=2026-10', 'anio_mes=2026-11']
temperatura  filas=2,880 rango=[-5.40, 23.40] particiones=['anio_mes=2026-07', 'anio_mes=2026-08', 'anio_mes=2026-09', 'anio_mes=2026-10', 'anio_mes=2026-11']


---
# 3 · Streaming: el flujo en vivo y el error de predicción en producción
Los consumers Spark corren en contenedores aparte. La ejecución integrada **observa una ventana fija** y comprueba que (1) llegan eventos nuevos de **ambas** dimensiones, (2) Spark los procesa y predice, y (3) el error del modelo a +1 h contra lo realmente observado es calculable.

In [5]:
def conteos():
    return sql('''SELECT (SELECT COUNT(*) FROM eventos_telemetria) hum, (SELECT COUNT(*) FROM eventos_temperatura) temp,
                         (SELECT COUNT(*) FROM predicciones_humedad) pred_hum, (SELECT COUNT(*) FROM predicciones_temperatura) pred_temp''').iloc[0].astype(int)
antes = conteos(); t0 = time.time()
print(f'Observando el flujo en vivo durante {VENTANA_STREAM_S} s (productores de 2 min + consumers Spark)…')
while time.time() - t0 < VENTANA_STREAM_S: time.sleep(10)
despues = conteos(); delta = (despues - antes).to_dict()
print('Filas nuevas en la ventana:', delta)
manifiesto['compuertas']['streaming'] = correr_compuertas({
  'Llegaron eventos nuevos de humedad (José)': lambda: delta['hum'] > 0,
  'Llegaron eventos nuevos de temperatura (Mijael)': lambda: delta['temp'] > 0,
  'Spark generó predicciones nuevas de humedad': lambda: delta['pred_hum'] > 0,
  'Spark generó predicciones nuevas de temperatura': lambda: delta['pred_temp'] > 0})
manifiesto['streaming'] = {'ventanaS': VENTANA_STREAM_S, 'filasNuevas': delta}
detener_si_falla(manifiesto['compuertas']['streaming'], 'streaming')

Observando el flujo en vivo durante 150 s (productores de 2 min + consumers Spark)…
Filas nuevas en la ventana: {'hum': 1, 'temp': 2, 'pred_hum': 3, 'pred_temp': 2}
OK    Llegaron eventos nuevos de humedad (José)
OK    Llegaron eventos nuevos de temperatura (Mijael)
OK    Spark generó predicciones nuevas de humedad
OK    Spark generó predicciones nuevas de temperatura


In [6]:
def error_prediccion(tabla_ev, col_obs, tabla_pred, col_pred):
    d = sql(f'''SELECT COUNT(*) n, AVG(ABS(p.{col_pred} - t.{col_obs})) mae_modelo, AVG(ABS(e.{col_obs} - t.{col_obs})) mae_persistencia
                FROM {tabla_pred} p JOIN {tabla_ev} t ON t.fecha_evento = p.fecha_objetivo JOIN {tabla_ev} e ON e.evento_id = p.evento_id
                WHERE p.horizonte_h = 1 AND t.origen IN {ORIGENES} AND e.origen IN {ORIGENES} AND p.fecha_objetivo >= NOW() - INTERVAL '24 hours' ''').iloc[0]
    return {k: (None if pd.isna(v) else float(v)) for k, v in d.items()}
err = {'humedad': error_prediccion('eventos_telemetria', 'humedad_suelo_pct', 'predicciones_humedad', 'prediccion_pct'),
       'temperatura': error_prediccion('eventos_temperatura', 'temperatura_c', 'predicciones_temperatura', 'prediccion_c')}
display(pd.DataFrame(err).T.round(4))
for dim, e in err.items():
    if not e['n']: manifiesto['advertencias'].append(f'[{dim}] aún no hay pares predicción/observación comparables (el objetivo +1 h todavía no ocurrió)')
    elif e['mae_modelo'] is not None and e['mae_persistencia'] is not None and e['mae_modelo'] >= e['mae_persistencia']:
        manifiesto['advertencias'].append(f'[{dim}] en producción el modelo NO supera a la persistencia (MAE {e["mae_modelo"]:.3f} ≥ {e["mae_persistencia"]:.3f})')
manifiesto['streaming']['errorPrediccion1h'] = err
for w in manifiesto['advertencias']: print('ADVERTENCIA:', w)

,n,mae_modelo,mae_persistencia
humedad,438.0,0.1031,0.0716
temperatura,439.0,1.0486,1.4096


ADVERTENCIA: [humedad] en producción el modelo NO supera a la persistencia (MAE 0.103 ≥ 0.072)


---
# 4 · Publicar en Prometheus (Pushgateway) y **confirmar el valor recién publicado**
Se registra la ejecución en PostgreSQL (`ejecuciones_integracion`, que el exportador expone como `agrosmart_int_*`) y se publican métricas en el Pushgateway (`agrosmart_ejec_*`). La confirmación compara **cada valor devuelto con el que acabas de publicar** (no basta con que *exista* un valor: podría ser el de una corrida anterior).

In [7]:
def num(x, d=0.0): return d if x is None else float(x)
metricas = {
 'agrosmart_ejec_latencia_p50_humedad_s': num(kpi['humedad']['p50']), 'agrosmart_ejec_latencia_p95_humedad_s': num(kpi['humedad']['p95']),
 'agrosmart_ejec_latencia_p50_temperatura_s': num(kpi['temperatura']['p50']), 'agrosmart_ejec_latencia_p95_temperatura_s': num(kpi['temperatura']['p95']),
 'agrosmart_ejec_eventos_nuevos_humedad': float(delta['hum']), 'agrosmart_ejec_eventos_nuevos_temperatura': float(delta['temp']),
 'agrosmart_ejec_error_1h_humedad': num(err['humedad']['mae_modelo']), 'agrosmart_ejec_error_1h_temperatura': num(err['temperatura']['mae_modelo']),
 'agrosmart_ejec_ultima_timestamp': float(int(time.time())),
}
cuerpo = ''.join(f'# TYPE {n} gauge\n{n} {v}\n' for n, v in metricas.items())
with urllib.request.urlopen(urllib.request.Request('http://pushgateway:9091/metrics/job/agrosmart_integracion/instance/u3', data=cuerpo.encode(), method='PUT'), timeout=10) as r:
    print('Pushgateway respondió:', r.status)
def consultar(nombre):
    res = json.load(urllib.request.urlopen('http://prometheus:9090/api/v1/query?query=' + nombre, timeout=10))['data']['result']
    return res[0]['value'][1] if res else None
def coincide(o, e): return o is not None and abs(float(o) - float(e)) <= 1e-6 * max(1.0, abs(float(e)))
conf, limite = {}, time.time() + 90
while time.time() < limite:
    conf = {n: consultar(n) for n in metricas}
    if all(coincide(conf[n], v) for n, v in metricas.items()): break
    time.sleep(5)
for n, v in metricas.items(): print(f'{n}: devuelto={conf[n]} | publicado={v}')
todas = all(coincide(conf[n], v) for n, v in metricas.items())
manifiesto['observabilidad'] = {'metricasPublicadas': metricas, 'confirmadasEnPrometheus': todas}
assert todas, 'Prometheus no devolvió los valores recién publicados'

Pushgateway respondió: 200
agrosmart_ejec_latencia_p50_humedad_s: devuelto=6.529086 | publicado=6.529086
agrosmart_ejec_latencia_p95_humedad_s: devuelto=9.424938999999993 | publicado=9.424938999999993
agrosmart_ejec_latencia_p50_temperatura_s: devuelto=6.1261305 | publicado=6.1261305
agrosmart_ejec_latencia_p95_temperatura_s: devuelto=7.31145215 | publicado=7.31145215
agrosmart_ejec_eventos_nuevos_humedad: devuelto=1 | publicado=1.0
agrosmart_ejec_eventos_nuevos_temperatura: devuelto=2 | publicado=2.0
agrosmart_ejec_error_1h_humedad: devuelto=0.10314775670216603 | publicado=0.10314775670216603
agrosmart_ejec_error_1h_temperatura: devuelto=1.0485629050434795 | publicado=1.0485629050434795
agrosmart_ejec_ultima_timestamp: devuelto=1795154730 | publicado=1795154730.0


In [8]:
manifiesto['estado'] = 'ADVERTENCIA' if manifiesto['advertencias'] else 'OK'
manifiesto['duracionTotalS'] = round(time.time() - INICIO, 1)
tot = sum(len(v) for v in manifiesto['compuertas'].values()); ok = sum(sum(v.values()) for v in manifiesto['compuertas'].values())
with psycopg2.connect(**PARAM_DB) as con, con.cursor() as cur:
    cur.execute('''INSERT INTO ejecuciones_integracion (run_id,estado,compuertas_ok,compuertas_total,duracion_s,lat_p50_hum_s,lat_p95_hum_s,lat_p50_temp_s,lat_p95_temp_s,
                   eventos_hum,eventos_temp,pred_error_hum_1h,pred_error_temp_1h,manifiesto) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                   ON CONFLICT(run_id) DO NOTHING''',
        (RUN_ID, manifiesto['estado'], ok, tot, manifiesto['duracionTotalS'], kpi['humedad']['p50'], kpi['humedad']['p95'], kpi['temperatura']['p50'], kpi['temperatura']['p95'],
         delta['hum'], delta['temp'], err['humedad']['mae_modelo'], err['temperatura']['mae_modelo'], json.dumps(manifiesto, default=str)))
print(f'Ejecución registrada: {RUN_ID} · estado {manifiesto["estado"]} · compuertas {ok}/{tot}')

Ejecución registrada: 20261120T060237Z · estado ADVERTENCIA · compuertas 24/24


---
# 5 · Manifiesto de ejecución y comparación de dos corridas
El manifiesto registra: identificador, versiones, **huellas (SHA-256) de la configuración**, resultado de cada compuerta, KPI y duración. Un pipeline con fuente en vivo **no es reproducible valor por valor**: lo reproducible es el **procedimiento**; lo auditable es **qué dio cada corrida**.

In [9]:
def huella(ruta):
    p = REPO/ruta; return hashlib.sha256(p.read_bytes()).hexdigest()[:16] if p.exists() else None
CONFIG = ['kafka/compose.yml', 'pyspark/compose.yml', 'pyspark/compose.kafka.yml', 'obs/compose.yml', 'uso-agrosmart/compose.yml', 'obs/prometheus/prometheus.yml',
          'obs/prometheus/alerts.yml', 'obs/grafana/dashboards/agrosmart_u2.json', 'pyspark/sql/schema.sql', 'pyspark/sql/schema_temperatura.sql', 'pyspark/sql/schema_integracion.sql']
manifiesto['huellasConfiguracion'] = {r: huella(r) for r in CONFIG}
manifiesto['versiones'] = {'spark': spark.version, 'psycopg2': psycopg2.__version__, 'pandas': pd.__version__}
ruta_m = EVID/f'manifiesto_{RUN_ID}.json'
ruta_m.write_text(json.dumps(manifiesto, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
print('Manifiesto escrito en', ruta_m); print(json.dumps({k: manifiesto[k] for k in ('runId', 'estado', 'duracionTotalS')}, indent=2))

Manifiesto escrito en /home/jovyan/work/U3_Integracion/evidencias/manifiesto_20261120T060237Z.json
{
  "runId": "20261120T060237Z",
  "estado": "ADVERTENCIA",
  "duracionTotalS": 183.7
}


In [10]:
def _g(d, *ks):
    for k in ks:
        if not isinstance(d, dict) or k not in d: return None
        d = d[k]
    return d

def comparar(ruta_a, ruta_b, tol_rel=0.5):
    A, B = (json.loads(Path(r).read_text(encoding='utf-8')) for r in (ruta_a, ruta_b)); filas = []
    for nom, M in (('A', A), ('B', B)):
        if M.get('estado') == 'FALLA':
            filas.append((f'estado de la corrida {nom}', '—', f"FALLA en la etapa «{M.get('etapaFalla')}»", 'la corrida se detuvo: no tiene KPI completos'))
    for etapa in ('previas', 'calidad', 'streaming'):
        a_et, b_et = (_g(A, 'compuertas', etapa) or {}), (_g(B, 'compuertas', etapa) or {})
        for n, ok in b_et.items():
            if a_et.get(n) and not ok: filas.append((f'compuerta · {n}', 'OK', 'FALLA', 'REGRESIÓN'))
    for dim in ('humedad', 'temperatura'):
        for k in ('p50', 'p95'):
            va, vb = _g(A, 'batch', 'kpiLatenciaPipeline', dim, k), _g(B, 'batch', 'kpiLatenciaPipeline', dim, k)
            if va is not None and vb is not None:
                cambio = abs(vb - va) / max(1e-9, abs(va)); filas.append((f'latencia {k} · {dim} (s)', round(va, 3), round(vb, 3), 'esperable (flujo en vivo)' if cambio <= tol_rel else 'REVISAR'))
        ea, eb = _g(A, 'streaming', 'errorPrediccion1h', dim, 'mae_modelo'), _g(B, 'streaming', 'errorPrediccion1h', dim, 'mae_modelo')
        if ea is not None and eb is not None: filas.append((f'error +1 h · {dim}', round(ea, 4), round(eb, 4), 'esperable (lecturas distintas)'))
    da, db = A.get('duracionTotalS'), B.get('duracionTotalS')
    if da is not None and db is not None: filas.append(('duración total (s)', da, db, 'esperable (carga del equipo)'))
    ha, hb = A.get('huellasConfiguracion'), B.get('huellasConfiguracion')
    if ha and hb:
        cfg = [r for r in CONFIG if ha.get(r) != hb.get(r)]
        filas.append(('archivos de configuración cambiados', '—', ', '.join(cfg) or 'ninguno', 'esperado si hubo cambios intencionales'))
    else:
        filas.append(('huellas de configuración', '—', 'no disponibles', 'una corrida se detuvo antes de calcularlas'))
    return pd.DataFrame(filas, columns=['comprobación', 'corrida A', 'corrida B', 'lectura'])

todos = sorted(EVID.glob('manifiesto_*.json'))
completos = [m for m in todos if not m.name.endswith('_FALLA.json')]
detenidos = [m.name for m in todos if m.name.endswith('_FALLA.json')]
print(f'Manifiestos completos: {len(completos)} · corridas detenidas por una compuerta: {len(detenidos)} {detenidos}')
if len(completos) >= 2: display(comparar(completos[-2], completos[-1]))
else: print('Aún hay menos de dos corridas COMPLETAS: ejecuta este cuaderno de arriba abajo hasta que termine sin FALLA y vuelve a esta celda.')
if detenidos and completos: print('\nComparación de la última corrida detenida con la última completa (muestra qué impidió terminar):'); display(comparar(EVID/detenidos[-1], completos[-1]))

Manifiestos completos: 4 · corridas detenidas por una compuerta: 1 ['manifiesto_20261117T061131Z_FALLA.json']


,comprobación,corrida A,corrida B,lectura
0,latencia p50 · humedad (s),3.67,6.529,REVISAR
1,latencia p95 · humedad (s),14.018,9.425,esperable (flujo en vivo)
2,error +1 h · humedad,0.1069,0.1031,esperable (lecturas distintas)
3,latencia p50 · temperatura (s),3.619,6.126,REVISAR
4,latencia p95 · temperatura (s),11.083,7.311,esperable (flujo en vivo)
5,error +1 h · temperatura,1.0782,1.0486,esperable (lecturas distintas)
6,duración total (s),185.8,183.7,esperable (carga del equipo)
7,archivos de configuración cambiados,—,ninguno,esperado si hubo cambios intencionales



Comparación de la última corrida detenida con la última completa (muestra qué impidió terminar):


,comprobación,corrida A,corrida B,lectura
0,estado de la corrida A,—,FALLA en la etapa «calidad»,la corrida se detuvo: no tiene KPI completos
1,latencia p50 · humedad (s),748.073,6.529,REVISAR
2,latencia p95 · humedad (s),30657.676,9.425,REVISAR
3,latencia p50 · temperatura (s),762.237,6.126,REVISAR
4,latencia p95 · temperatura (s),30671.839,7.311,REVISAR
5,duración total (s),7.2,183.7,esperable (carga del equipo)
6,huellas de configuración,—,no disponibles,una corrida se detuvo antes de calcularlas


---
# 6 · Una compuerta que falla a propósito (demostración)
Se simula una dependencia ausente para mostrar que la ejecución **se detiene en el acto** con el nombre de lo que falta, sin tocar Spark.

In [11]:
demo = {'Modelo de Mijael existe (simulado: ruta inexistente)': lambda: (RAIZ/'Sensortemperatura'/'modelos'/'no_existe_2h').exists(),
        'PostgreSQL responde': lambda: bool(sql('SELECT 1 AS n').n.iloc[0])}
res_demo = correr_compuertas(demo)
try: detener_si_falla(res_demo, 'demostración', registrar=False)       # registrar=False: la demo no se guarda como corrida real
except AssertionError as e: print('\n→ La ejecución se detuvo como debe:', e)
manifiesto['demoCompuertaFallando'] = {'resultado': res_demo}

FALLA Modelo de Mijael existe (simulado: ruta inexistente)
OK    PostgreSQL responde

→ La ejecución se detuvo como debe: [demostración] Compuertas que fallan: ['Modelo de Mijael existe (simulado: ruta inexistente)']


---
# 7 · S14 — Revisión técnica final y hardening
## 7.1 Lista de comprobación de configuración (Tabla 2 de S14) — aplicada a los 5 archivos de Compose
La auditoría es el script `scripts/auditoria_config.py` (la misma que corre en CI y como prueba de no regresión).

In [12]:
sys.path.insert(0, str(REPO/'scripts'))
import importlib, auditoria_config; importlib.reload(auditoria_config)
aud = pd.DataFrame(auditoria_config.auditar()); display(aud)
(EVID/f'auditoria_config_{RUN_ID}.json').write_text(aud.to_json(orient='records', force_ascii=False, indent=2), encoding='utf-8')
assert not (aud.resultado == 'Falla').any(), 'Hay puntos de la lista de comprobación que fallan'

,punto,resultado,detalle
0,Versión de las imágenes,Cumple,Todas las imágenes tienen versión exacta.
1,Reinicio (restart),Cumple,Los servicios permanentes declaran restart.
2,Salud (healthcheck),Cumple,Healthcheck en 9 servicios permanentes.
3,Salud · riesgo aceptado: kafka-ui,Aceptado,Interfaz de apoyo; la imagen no garantiza util...
4,Salud · riesgo aceptado: kafka-exporter,Aceptado,Imagen mínima sin shell; su estado se observa ...
5,Orden de arranque (depends_on),Cumple,Toda dependencia con healthcheck espera a serv...
6,Secretos fuera del archivo,Cumple,Credenciales por ${VARIABLE}; .env en .gitignore.
7,Superficie expuesta (ports),Cumple,"11 puertos publicados, todos en 127.0.0.1 (sol..."
8,Variables documentadas en .env.example,Cumple,"6 variables obligatorias, todas en .env.example."


## 7.2 Bitácora de la revisión técnica (hallazgos corregidos y su prueba)
| # | Hallazgo | Causa | Corrección | Prueba de corrección (no regresión) |
|---|---|---|---|---|
| 1 | Ningún servicio declaraba si estaba **listo**; las dependencias esperaban solo a que el contenedor *arrancara*. | Compose escritos con configuración mínima de clase. | `healthcheck` en `kafka`, `exporter`, `prometheus`, `pushgateway`, `grafana`, `jupyter` y los dos consumers Spark (heartbeat en `metricas_pipeline`); `depends_on` con `service_healthy`. | `scripts/auditoria_config.py` → «Salud» y «Orden de arranque» = Cumple; `docker ps` muestra `(healthy)`; `up -d --wait` termina con código 0. |
| 2 | El Pushgateway estaba aislado de la red de Jupyter y su job no conservaba etiquetas. | Solo en `obs-net`; sin `honor_labels`. | Se conecta a `data-net` y se agrega `honor_labels: true`. | La celda 4 publica y Prometheus devuelve el **mismo valor** recién publicado. |
| 3 | No existía registro auditable de cada corrida integrada. | Los notebooks se ejecutaban sueltos. | Tabla `ejecuciones_integracion` + manifiesto JSON con huellas de configuración + métricas `agrosmart_int_*` y alertas. | Se ejecuta dos veces y `comparar()` clasifica cada diferencia como esperable o regresión. |
| 4 | Variables sin documentar para el despliegue del equipo (Mijael). | Faltaban en `.env.example`. | Se agregan `TEMP_LAT`, `TEMP_LON`, `TEMP_UBICACION`, `UMBRAL_HELADA_C` (opcionales). | Auditoría «Variables documentadas» = Cumple. |

| 5 | **Predicciones guardadas con 5 h de desfase**: la hora objetivo quedaba antes del propio evento y todas salían `es_retro = true`; el error en vivo comparaba horas distintas. | PySpark entrega los timestamps a Python sin zona (hora local del contenedor, `TZ=America/Lima`) y el consumer los etiquetaba como UTC. | `hora_peru()` en ambos consumers, conexión PostgreSQL con `TimeZone=America/Lima`, PostgreSQL y Grafana en hora de Perú y `reparar_hora_peru.sql` (desfase medido: 18 001 s; 1 346 + 1 262 lecturas corregidas). | `verificacion.sql`: emisión − evento = 1 s y 120 s; objetivo = evento + h; `es_retro = f`; esta corrida comparada con la del 17-nov. |
| 6 | El consumer de verificación rechazaba el tópico de temperatura y leía desde el offset 0. | `choices` solo de humedad. | Acepta los 4 tópicos y muestra los últimos N mensajes. | `verificar_sistema.bat`, sección 4. |
| 7 | Evidencias, resultados y modelos excluidos de git. | `.gitignore` demasiado amplio. | Se versionan; siguen fuera `.env`, checkpoints y temporales. | `git ls-files`. |
| 8 | Tablero con tablas desbordadas y métricas de los dos integrantes mezcladas. | Dashboard crecido por partes. | Dashboard v4 en 3 partes y Prometheus filtrado por consulta. | `test_hardening` y revisión visual. |

## 7.3 Prueba de no regresión
Tras **cada cambio de configuración**: (1) `python -m unittest discover -s pruebas -v` (20+ pruebas, incluye `test_hardening.py`), (2) `python scripts/auditoria_config.py`, (3) **repetir este cuaderno** y comparar manifiestos (celda de la sección 5). Que cambie el error del modelo o la latencia **no** es una regresión (la fuente es en vivo); que una compuerta pase de OK a FALLA, o que el flujo procese cero filas, **sí** lo es.

## 7.4 Riesgos aceptados (justificación y qué cambiaría en producción)
| Riesgo | Por qué se acepta aquí | Qué cambiaría en producción |
|---|---|---|
| Kafka en `PLAINTEXT` y sin autenticación, un solo broker, replicación 1. | Laboratorio local (puertos en `127.0.0.1`); TLS/SASL complica cada ejemplo sin aportar al objetivo de la unidad. | TLS + SASL, ≥ 3 brokers, replicación 3. |
| Credenciales de laboratorio en `.env` (no versionado) y token de Jupyter simple. | Entorno local de un estudiante; el `.env` está en `.gitignore` y `.env.example` solo lleva marcadores. | Gestor de secretos y rotación; Jupyter detrás de un acceso autenticado. |
| Sin healthcheck en `kafka-ui` y `kafka-exporter`. | La imagen mínima no garantiza shell; el exportador se observa desde Prometheus (target UP/DOWN) y la UI no afecta al flujo. | Sondas HTTP desde el orquestador. |
| Datos modelados (Open-Meteo) e interpolados/simulados; no sensores físicos. | No existe API gratuita con humedad del suelo observada cada 2 min para la zona; cada fila lleva `origen`. | Sensores físicos (ESP32) publicando a Kafka con el mismo contrato. |
| Un solo nodo Spark (`local[2]`) por consumer. | Volumen bajo (1 evento/2 min por dimensión). | Clúster Spark con checkpoints en almacenamiento compartido. |
| Umbral de helada (0 °C) y de humedad (20 %) didácticos; costos en S/ con tarifas supuestas. | Sirven para demostrar el flujo de decisión, no son recomendaciones agronómicas ni precios comerciales. | Umbrales validados con un agrónomo y tarifas reales del proveedor. |

## 7.5 Guion de demostración (10 min) con plan alternativo
| Min | Qué se muestra | Comando o URL | Qué debe verse | Plan alternativo |
|---|---|---|---|---|
| 0–1 | Entorno sano | `docker ps` | Servicios `(healthy)` | Captura con fecha y hora |
| 1–2 | Compuertas previas | Sección 1 de este cuaderno | 10 × OK | Manifiesto de una corrida anterior |
| 2–4 | Flujo cada 2 min (José y Mijael) | `docker compose … logs -f producer-humedad-2min producer-temperatura-2min` + Kafka UI | Offsets creciendo | Salida guardada |
| 4–5 | Batch: KPI p50/p95 y Gold | Sección 2 | p50 frente a p95 y advertencias | Captura de 2.1 |
| 5–7 | Streaming, predicción +1 h y error en producción | Sección 3 + fila 01–07 de Grafana | Filas nuevas y error | Captura del tablero |
| 7–8 | Prometheus confirma lo publicado | Sección 4 | `devuelto = publicado` | Consulta manual en `:9095` |
| 8–9 | Manifiesto y comparación de 2 corridas | Sección 5 | Tabla esperable/regresión | JSON abierto |
| 9–10 | Hardening | Sección 7.1 y `pruebas/test_hardening.py` | Todos «Cumple/Aceptado» | `python scripts/auditoria_config.py` |

**Antes de empezar:** verifica que los productores de 2 minutos estén publicando (los offsets del tópico deben crecer).

## 7.6 Evidencias a capturar (con reloj del sistema y usuario visibles, sin recortar)
Compuertas previas (1) · una compuerta fallando (6) · KPI p50/p95 (2) · Prometheus devolviendo lo publicado (4) · dos manifiestos y su comparación (5) · lista de comprobación (7.1) · `docker ps` con `(healthy)` · tablero Grafana completo.